# 08. 실제 이미지 데이터로 MobileNetV3-Small 파인튜닝

합성 센서 MLP는 model contract와 배포 자동화를 빠르게 배우는 smoke test다. 이 실습은 실제 로봇 카메라 이미지로 사전학습 경량 모델을 파인튜닝한다.

**목표**

- train/val/test를 mission 단위로 먼저 분리한다.
- ImageNet 사전학습 MobileNetV3-Small의 전처리를 보존한다.
- 새 classifier head만 학습한 뒤 전체 backbone을 작은 learning rate로 푼다.
- 독립 test의 전체 accuracy와 class별 recall을 계산한다.
- RGB/NCHW/mean/std/class map을 metadata와 ONNX로 저장한다.

전체 코드는 `src/vision_fine_tune.py`에 있으며 모든 실행문 위에 한국어 설명이 있다.

## 데이터 폴더

아래 예시는 장애물 상태 분류다. 실제 class 이름으로 교체한다.

```text
data/robot_vision/
  train/
    clear/
    blocked/
    unknown/
  val/
    clear/
    blocked/
    unknown/
  test/
    clear/
    blocked/
    unknown/
```

같은 동영상의 frame을 무작위로 세 폴더에 나누지 않는다. `robot_id + mission_id + site + date`를 group key로 먼저 나눈 뒤 이미지를 배치한다.

## 1. 폴더별 이미지 수와 class 일치를 검사한다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | `Path` import | 운영체제 독립 경로를 사용한다. |
| 2 | root 객체 | dataset root를 한 곳에서 바꾼다. |
| 3 | nested comprehension | split과 class별 파일 수를 dictionary로 만든다. |
| 4 | 출력 | class 누락과 심한 imbalance를 학습 전에 찾는다. |

In [ ]:
# Path는 `/` 연산자로 폴더 이름을 안전하게 결합한다.
from pathlib import Path
# 실제 dataset root를 지정한다.
data_root = Path('data/robot_vision')
# 각 split의 class 폴더와 파일 수를 dictionary로 계산한다.
counts = {split: {folder.name: len([path for path in folder.rglob('*') if path.is_file()]) for folder in sorted((data_root / split).iterdir()) if folder.is_dir()} for split in ('train', 'val', 'test')}
# class 누락, 0건, imbalance가 없는지 사람이 검토한다.
counts

## 2. 두 단계 파인튜닝을 실행한다

첫 실행 때 torchvision이 공식 사전학습 weight를 내려받을 수 있다. 제품 빌드에서는 승인된 weight checksum을 artifact store에 고정한다.

| 인수 | 뜻 |
|---|---|
| `--data-dir` | train/val/test가 있는 root |
| `--head-epochs` | backbone을 얼리고 새 head만 학습 |
| `--full-epochs` | 전체 model을 작은 learning rate로 미세조정 |
| `--workers` | image decode/transform process 수; notebook 문제 시 0으로 시작 |
| `--artifact-dir` | checkpoint, ONNX, metadata 출력 폴더 |

In [ ]:
# %run은 상세 주석이 있는 실제 training script를 현재 kernel에서 실행한다.
%run src/vision_fine_tune.py --data-dir data/robot_vision --artifact-dir artifacts/vision --head-epochs 5 --full-epochs 5 --batch-size 32 --workers 0

## 3. metadata를 검토한다

| 줄 | 문법 | 이유 |
|---|---|---|
| 1 | json import | metadata text를 dictionary로 읽는다. |
| 2 | context manager | UTF-8 file을 자동으로 닫는다. |
| 3 | `json.load` | class map과 전처리 계약을 읽는다. |
| 4 | pretty print | RGB, NCHW, mean/std, class recall을 검토한다. |

In [ ]:
# json은 metadata를 Python dictionary로 변환한다.
import json
# UTF-8 metadata를 읽고 cell 종료 때 자동으로 닫는다.
with Path('artifacts/vision/metadata.json').open('r', encoding='utf-8') as file:
    # load가 ONNX 옆의 전처리와 class mapping을 읽는다.
    metadata = json.load(file)
# C++ 구현 전에 모든 계약과 class별 recall을 검토한다.
print(json.dumps(metadata, ensure_ascii=False, indent=2))

## 실무 확장 과제

1. augmentation을 추가하되 좌우 의미, 카메라 geometry, motion blur가 물리적으로 가능한지 검토한다.
2. class imbalance에 weighted loss와 sampler를 각각 적용해 위험 class recall을 비교한다.
3. 세 seed를 실행하고 평균·표준편차를 보고한다.
4. confidence reliability diagram과 expected calibration error를 추가한다.
5. FP16과 INT8로 변환해 target의 accuracy/p99/power/temperature를 비교한다.
6. golden image 100장을 Python, ONNX Runtime, C++에서 비교한다.
7. glare, rain, blur, occlusion, dirty lens를 challenge set으로 분리한다.

**금지**: test 결과를 보고 epoch, augmentation, threshold를 계속 바꾸지 않는다. 그 순간 test는 validation이 된다.